### **Student Information**
GitHub ID: Nich06

Kaggle name: nicholashhh06

Kaggle private scoreboard snapshot: this is a late submission made after the competition closed on November 22, so it does not appear on the final private leaderboard. The screenshot shows the private score Kaggle assigned to it on the Submissions page.

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

---
# Project Report
---

## 1. Model Development

### 1.1 Preprocessing Steps
- **Loading:** I joined `final_posts.json` with `data_identification.csv` (train/test split) and `emotion.csv` (labels) on `post_id`. This gave 3,360 labelled training posts and 1,440 test posts across six emotions.
- **Cleaning:** I replaced the anonymised `@<username>` with one consistent token (`@user`) and collapsed extra whitespace and line breaks. I kept the `[NAME]` placeholder, since the tokenizer handles it consistently.
- **Duplicates:** I removed exact duplicate rows from the training set. The test set was left untouched.
- **Validation split:** I held out 15% of the training data (stratified by label) to measure macro F1 and choose the best epoch.
- **Class imbalance:** the classes are uneven (joy is the largest, fear the smallest), which is why I used macro F1 to judge every step.

### 1.2 Feature Engineering Steps
I used the pre-trained tokenizer that belongs to the model. It splits each post into subword tokens (Ex: `Ġlike` marks the word "like"), so rare or misspelled words are still broken into known pieces. Each post was cut to at most hundred tokens, which covers almost every post. The model then learns its own features from these tokens during fine-tuning.

### 1.3 Explanation of Your Model
I fine-tuned **`j-hartmann/emotion-english-distilroberta-base`**, a small transformer that had already been trained to recognise seven emotions (our six plus *neutral*). I kept its existing output weights for our six emotions and dropped *neutral*, so the model started from emotion knowledge instead of from scratch.

- **Training:** 4 epochs, batch size 16, learning rate 2e-5 (AdamW), a short warm-up followed by linear decay, on the Apple GPU of my Macbook M5 chip laptop.
- **Model selection:** after each epoch, I measured validation macro F1 and kept the best epoch.

| Stage | Validation macro F1 |
|---|---|
| Before fine-tuning | 0.5616 |
| Epoch 1 / 2 / 3 / 4 | 0.5889 / 0.5968 / **0.6095** / 0.6072 |

**Kaggle result:** private score **0.6168**, public score **0.6238**, above the 0.5 baseline. This is a **late submission**: I submitted after the competition closed on November 22, so it does not appear on the final leaderboard.

**Interpretation:** fine-tuning added about 0.05 macro F1 over the pre-trained model, and the score stopped improving after epoch 3, so more epochs would mainly overfit. The validation and Kaggle scores are close, which suggests the validation split was a fair estimate of unseen data.

### 1.4 AI Prompts, what and why you decided to use the technique suggested by AI
I used Claude throughout this part to re-exam my logic as a third party between me and agentic AI. So this project have 3 Points of view make it more accurate and reliable.
Ex:
| "The upload submission failed, is this have something to do with my late?" | then AI suggested renaming the column `post_id` to `id`. | **Accepted:** the second upload was scored successfully. |

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [4]:
### Preprocessing: load the competition files, clean the text, split train/validation
import re, numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"   # Apple GPU on the M5
print("device:", DEVICE)

DATA = "data/kaggle/"
LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
lab2id = {l: i for i, l in enumerate(LABELS)}

# Join the posts with their train/test membership and the training labels.
posts = pd.read_json(DATA + "final_posts.json")
ids = pd.read_csv(DATA + "data_identification.csv")
labels = pd.read_csv(DATA + "emotion.csv")
df = posts.merge(ids, on="post_id").merge(labels, on="post_id", how="left")

def clean(s):
    s = s.replace("@<username>", "@user")       # one consistent token for anonymised users
    return re.sub(r"\s+", " ", s).strip()       # collapse whitespace and line breaks
df["text"] = df["text"].map(clean)

# Drop exact duplicate (text, label) rows; keep the test set untouched.
train = df[df["set"] == "train"].drop_duplicates(["text", "label"]).reset_index(drop=True)
test = df[df["set"] == "test"].reset_index(drop=True)
# Hold out 15% of train, stratified, to measure macro F1 and pick the best epoch.
tr, va = train_test_split(train, test_size=0.15, stratify=train["label"], random_state=SEED)
print("train / validation / test:", len(tr), len(va), len(test))
print(tr["label"].value_counts().to_dict())

device: mps
train / validation / test: 2827 499 1440
{'joy': 740, 'anger': 567, 'sadness': 526, 'disgust': 383, 'surprise': 371, 'fear': 240}


## 2. Feature Engineering Steps

In [5]:
### Feature engineering: subword tokens from the model's own pre-trained tokenizer
MODEL_NAME = "j-hartmann/emotion-english-distilroberta-base"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

def encode(texts):
    # 128 tokens covers almost every post (95% have 32 words or fewer); longer ones are cut.
    enc = tok(list(texts), truncation=True, max_length=128, padding=True, return_tensors="pt")
    return {k: v.to(DEVICE) for k, v in enc.items()}

# Show what the model actually receives for one post.
example = tr["text"].iloc[0]
print(example)
print(tok.tokenize(example)[:20])

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

You like [NAME] but dislike [NAME]? That is weird considering that they both are on the same low level of intelligence.
['You', 'Ġlike', 'Ġ[', 'NAME', ']', 'Ġbut', 'Ġdislike', 'Ġ[', 'NAME', ']', '?', 'ĠThat', 'Ġis', 'Ġweird', 'Ġconsidering', 'Ġthat', 'Ġthey', 'Ġboth', 'Ġare', 'Ġon']


## 3. Model Implementation Steps

In [6]:
### Model: fine-tune an emotion-pretrained RoBERTa and write the submission file
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
# Reuse the pretrained output rows for our six emotions and drop 'neutral'.
src = {v.lower(): int(k) for k, v in model.config.id2label.items()}
rows = [src[l] for l in LABELS]
old = model.classifier.out_proj
new = torch.nn.Linear(old.in_features, len(LABELS))
with torch.no_grad():
    new.weight.copy_(old.weight[rows]); new.bias.copy_(old.bias[rows])
model.classifier.out_proj = new
model.to(DEVICE)

@torch.no_grad()
def predict_logits(texts, bs=64):
    model.eval()
    texts = list(texts)
    return torch.cat([model(**encode(texts[i:i + bs])).logits.float().cpu()
                      for i in range(0, len(texts), bs)])

def macro_f1(frame):
    pred = predict_logits(frame["text"]).argmax(1).numpy()
    return f1_score(frame["label"].map(lab2id), pred, average="macro")

print("before fine-tuning, validation macro F1:", round(macro_f1(va), 4))

EPOCHS, BS, LR = 4, 16, 2e-5
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * int(np.ceil(len(tr) / BS)); warm = int(0.1 * steps)
# Linear warm-up for 10% of the steps, then linear decay to zero.
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (steps - s) / (steps - warm)))
loss_fn = torch.nn.CrossEntropyLoss()

texts_tr, y_tr = tr["text"].tolist(), tr["label"].map(lab2id).to_numpy()
best_f1, best_state = -1.0, None
for epoch in range(EPOCHS):
    model.train()
    order = np.random.permutation(len(tr))
    for i in range(0, len(order), BS):
        b = order[i:i + BS]
        logits = model(**encode([texts_tr[j] for j in b])).logits
        loss = loss_fn(logits, torch.tensor(y_tr[b], device=DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
    f1 = macro_f1(va)
    print(f"epoch {epoch + 1}: validation macro F1 {f1:.4f}")
    if f1 > best_f1:                              # keep the best epoch, not the last one
        best_f1 = f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print("best validation macro F1:", round(best_f1, 4))

# Predict the test set and write the file in the sampleSubmission.csv format.
test_pred = predict_logits(test["text"]).argmax(1).numpy()
pd.DataFrame({"post_id": test["post_id"], "label": [LABELS[i] for i in test_pred]}) \
  .to_csv("submission.csv", index=False)
print(pd.read_csv("submission.csv")["label"].value_counts())

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  329MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

before fine-tuning, validation macro F1: 0.5616
epoch 1: validation macro F1 0.5889
epoch 2: validation macro F1 0.5968
epoch 3: validation macro F1 0.6095
epoch 4: validation macro F1 0.6072
best validation macro F1: 0.6095
label
joy         395
anger       292
sadness     262
disgust     204
surprise    182
fear        105
Name: count, dtype: int64


In [7]:
# Kaggle's scorer expects the column "id" (as on the Evaluation page), not "post_id".
sub = pd.read_csv("submission.csv")
sub.rename(columns={"post_id": "id"}).to_csv("submission_id.csv", index=False)
print(pd.read_csv("submission_id.csv").head())

        id     label
0  xyzx9jp  surprise
1  l3vzzxt   sadness
2  g680iug   sadness
3  m4t8bp6       joy
4  ai3al6e  surprise
